In [1]:
print("Bonjour Sysim Guard")



Bonjour Sysim Guard


In [2]:
import os
os.listdir()


['.ipynb_checkpoints',
 'exploration_krobodroid.ipynb',
 'real_legitimate_v1.csv',
 'real_legitimate_v1.zip',
 'real_malware_v1.csv',
 'real_malware_v1.zip']

In [5]:
import pandas as pd
sains = pd.read_csv("real_legitimate_v1.csv", low_memory=False)
malwares = pd.read_csv("real_malware_v1.csv", low_memory=False)
print("Applications saines :", sains.shape)
print("Malwares :", malwares.shape)

Applications saines : (36755, 484)
Malwares : (41382, 484)


In [6]:
print(list(sains.columns) == list(malwares.columns))

True


In [7]:
pd.Series(malwares.columns).to_csv("colonnes.csv", index=False)
print(list(malwares.columns[-40:]))

['USE_SIP', 'VIBRATE', 'WAKE_LOCK', 'WRITE_APN_SETTINGS', 'WRITE_CALENDAR', 'WRITE_CALL_LOG', 'WRITE_CONTACTS', 'WRITE_EXTERNAL_STORAGE', 'WRITE_GSERVICES', 'WRITE_SECURE_SETTINGS', 'WRITE_SETTINGS', 'WRITE_SYNC_SETTINGS', 'WRITE_VOICEMAIL', 'nr_permissions', 'normal', 'dangerous', 'signature', 'custom_yes', 'nr_custom', 'total_perm', 'sha256', 'CFileSize', 'UFileSize', 'EarliestModDate', 'HighestModDate', 'FilesInsideAPK', 'TimesSubmitted', 'NrContactedIps', 'Activities', 'NrIntServices', 'NrIntServicesActions', 'NrIntActivities', 'NrIntActivitiesActions', 'NrIntReceivers', 'NrIntReceiversActions', 'TotalIntentFilters', 'NrServices', 'Scanners', 'Detection_Ratio', 'MalFamily']


In [8]:
print(list(malwares.columns[:30]))
print([c for c in malwares.columns if any(m in c.lower() for m in ["date", "seen", "first", "last", "malware", "label", "class"])])
permissions = [c for c in malwares.columns if c.isupper()]
print("Nombre de colonnes de permissions :", len(permissions))

['Package', 'Malware', 'execve', 'getuid32', 'getgid32', 'geteuid32', 'getegid32', 'getresuid32', 'getresgid32', 'readahead', 'getgroups32', 'getpgid', 'getppid', 'getsid', 'setsid', 'setgid32', 'setuid32', 'setreuid32', 'setresuid32', 'setresgid32', 'brk', 'kill', 'tgkill', 'ptrace', 'getrusage', 'getpriority', 'setpriority', 'ugetrlimit', 'setrlimit', 'prlimit64']
['Malware', 'CONTROL_LOCATION_UPDATES', 'UPDATE_DEVICE_STATS', 'EarliestModDate', 'HighestModDate']
Nombre de colonnes de permissions : 236


In [9]:
df = pd.concat([sains, malwares], ignore_index=True)
print(df["Malware"].value_counts())
print("Permissions en 0/1 :", df[permissions].isin([0, 1]).all().all())
df["annee"] = pd.to_datetime(df["HighestModDate"], errors="coerce").dt.year
pd.crosstab(df["annee"], df["Malware"])

Malware
1    41382
0    36755
Name: count, dtype: int64
Permissions en 0/1 : False


C:\Windows\Temp\ipykernel_21964\2705413412.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["annee"] = pd.to_datetime(df["HighestModDate"], errors="coerce").dt.year


Malware,0,1
annee,,
1980.0,8,16
1981.0,27,0
1985.0,4,0
2004.0,1,0
2005.0,0,1
2008.0,64,968
2009.0,623,20
2010.0,5084,269
2011.0,23089,3137


In [10]:
df = df.copy()
pas_binaires = [c for c in permissions if not df[c].isin([0, 1]).all()]
print("Colonnes concernées :", pas_binaires)
for c in pas_binaires[:10]:
    print(c, "→ valeurs :", df[c].unique()[:10])

Colonnes concernées : ['SYS_300', 'SYS_301', 'SYS_305', 'SYS_306', 'SYS_307', 'SYS_310', 'SYS_311', 'SYS_312', 'SYS_314', 'SYS_315', 'SYS_316', 'SYS_317', 'SYS_318', 'SYS_319', 'SYS_328', 'SYS_329', 'SYS_331', 'SYS_335', 'SYS_336', 'SYS_339', 'SYS_340', 'SYS_341', 'SYS_342', 'SYS_359', 'SYS_367']
SYS_300 → valeurs : [0 2 1 7 5]
SYS_301 → valeurs : [0 1 2]
SYS_305 → valeurs : [44  0  4 19  3 28 24  1 15 21]
SYS_306 → valeurs : [ 4  0  3  5  6  1  2 13 10  7]
SYS_307 → valeurs : [ 0 12 54  5 15  8  3 10 21  2]
SYS_310 → valeurs : [130   0  36  20  39  21 141 123  64   9]
SYS_311 → valeurs : [ 1  0  9  2  3  8  4 19 11 10]
SYS_312 → valeurs : [ 0  8  1  3  7  4  9  2 26 10]
SYS_314 → valeurs : [ 0  3 25]
SYS_315 → valeurs : [ 5  0  2 10  6  4  1 68  3  8]


In [11]:
avant = len(df)
df = df[(df["annee"] >= 2008) & (df["annee"] <= 2020)]
print("Applications retirées :", avant - len(df))
print(df["Malware"].value_counts())

Applications retirées : 1421
Malware
1    40934
0    35782
Name: count, dtype: int64


In [12]:
permissions = [c for c in permissions if not c.startswith("SYS_")]
print("Vraies permissions :", len(permissions))
print("Permissions en 0/1 :", df[permissions].isin([0, 1]).all().all())
print(permissions[:15])

Vraies permissions : 166
Permissions en 0/1 : True
['ACCEPT_HANDOVER', 'ACCESS_BACKGROUND_LOCATION', 'ACCESS_CHECKIN_PROPERTIES', 'ACCESS_COARSE_LOCATION', 'ACCESS_FINE_LOCATION', 'ACCESS_LOCATION_EXTRA_COMMANDS', 'ACCESS_MEDIA_LOCATION', 'ACCESS_NETWORK_STATE', 'ACCESS_NOTIFICATION_POLICY', 'ACCESS_WIFI_STATE', 'ACCOUNT_MANAGER', 'ACTIVITY_RECOGNITION', 'ADD_VOICEMAIL', 'ANSWER_PHONE_CALLS', 'BATTERY_STATS']


In [14]:
compteurs = ["nr_permissions", "normal", "dangerous", "signature", "custom_yes", "nr_custom", "total_perm"]
colonnes_gardees = permissions + compteurs + ["Malware", "annee", "HighestModDate", "sha256"]
df[colonnes_gardees].to_csv("kronodroid_permissions.csv", index=False)
print("Fichier enregistré :", df[colonnes_gardees].shape)

Fichier enregistré : (76716, 177)
